In [57]:
# --- One cell: same code for Snowflake Notebook and local Jupyter ---

from snowflake.snowpark import Session
import os

# Edit these once (used for local fallback + context standardization)
SF = {
    "account": "SFCOGSOPS-SNOWHOUSE_AWS_US_WEST_2",
    "user": "AALUSI",
    "authenticator": "externalbrowser",
    "warehouse": "AGAVIC_WH",
    "role": "PUBLIC",
    "database": "SNOWPUBLIC",
    "schema": "NOTEBOOKS",
}

def get_session():
    # 1) Try Snowflake Notebook session
    try:
        from snowflake.snowpark.context import get_active_session
        s = get_active_session()
        # Probe: if we can run a trivial query, it's real; otherwise fall back
        try:
            s.sql("SELECT 1").collect()
            return s
        except Exception:
            pass
    except Exception:
        pass

    # 2) Local Jupyter fallback
    return Session.builder.configs(SF).create()

session = get_session()

# Standardize context (applies in both environments)
for cmd in (
    f"USE ROLE {SF['role']}",
    f"USE WAREHOUSE {SF['warehouse']}",
    f"USE DATABASE {SF['database']}",
    f"USE SCHEMA {SF['schema']}",
):
    try:
        session.sql(cmd).collect()
    except Exception as e:
        print(f"⚠️ {cmd} -> {e}")

# Smoke check
session.sql(
    "SELECT current_role() role, current_warehouse() wh, current_database() db, current_schema() sch"
).show()

In [58]:
%pip install pandas pyarrow
%pip install matplotlib

In [59]:

df = session.sql("SELECT * FROM samples.tickit.sales LIMIT 1000").to_pandas()
df.head()

In [60]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# ---- 0) Copy + basic dtype cleanup ----
df2 = df.copy()

# Ensure correct types (safe even if already correct)
for col in ["QTYSOLD", "PRICEPAID", "COMMISSION"]:
    df2[col] = pd.to_numeric(df2[col], errors="coerce")
df2["SALETIME"] = pd.to_datetime(df2["SALETIME"], errors="coerce")

# ---- 1) Derived fields ----
df2["GROSS_SALE"] = df2["PRICEPAID"]                    # total paid
df2["NET_SALE"]   = df2["PRICEPAID"] - df2["COMMISSION"]# after commission
df2["UNIT_PRICE"] = df2["PRICEPAID"] / df2["QTYSOLD"]
df2["SALE_DATE"]  = df2["SALETIME"].dt.date
df2["WEEK_START"] = df2["SALETIME"].dt.to_period("W").dt.start_time

# ---- 2) Core summaries (display these) ----
daily = (df2.groupby("SALE_DATE")
           .agg(orders=("SALESID","count"),
                qty=("QTYSOLD","sum"),
                gross=("GROSS_SALE","sum"),
                net=("NET_SALE","sum"))
           .reset_index())

top_sellers = (df2.groupby("SELLERID")
                 .agg(orders=("SALESID","count"),
                      qty=("QTYSOLD","sum"),
                      gross=("GROSS_SALE","sum"),
                      net=("NET_SALE","sum"),
                      avg_unit_price=("UNIT_PRICE","mean"))
                 .sort_values("gross", ascending=False)
                 .head(10)
                 .reset_index())

top_events = (df2.groupby("EVENTID")
                .agg(orders=("SALESID","count"),
                     qty=("QTYSOLD","sum"),
                     gross=("GROSS_SALE","sum"),
                     net=("NET_SALE","sum"))
                .sort_values("gross", ascending=False)
                .head(10)
                .reset_index())

price_qty_corr = df2[["QTYSOLD","UNIT_PRICE"]].corr().loc["QTYSOLD","UNIT_PRICE"]

print("📊 Daily summary:")
display(daily.head(10))

print("\n🏷️  Top sellers (by gross):")
display(top_sellers)

print("\n🎫  Top events (by gross):")
display(top_events)

print(f"\n🔗 Correlation (Qty vs Unit Price): {price_qty_corr:.3f}")

# ---- 3) Quick charts (matplotlib; no seaborn) ----

# 3a) Net revenue over time
plt.figure(figsize=(8,4))
plt.plot(daily["SALE_DATE"], daily["net"], marker="o")
plt.title("Net Revenue by Day")
plt.xlabel("Date")
plt.ylabel("Net Revenue")
plt.grid(True)
plt.tight_layout()
plt.show()

# 3b) Distribution of unit prices
plt.figure(figsize=(8,4))
df2["UNIT_PRICE"].dropna().plot(kind="hist", bins=20)
plt.title("Distribution of Unit Price")
plt.xlabel("Unit Price")
plt.ylabel("Count")
plt.grid(True)
plt.tight_layout()
plt.show()

# 3c) Quantity vs Unit Price scatter
plt.figure(figsize=(6,6))
plt.scatter(df2["QTYSOLD"], df2["UNIT_PRICE"])
plt.title("Qty vs Unit Price")
plt.xlabel("Quantity Sold")
plt.ylabel("Unit Price")
plt.grid(True)
plt.tight_layout()
plt.show()

# ---- 4) Optional: weekly view (nice for longer series) ----
weekly = (df2.groupby("WEEK_START")
            .agg(orders=("SALESID","count"),
                 qty=("QTYSOLD","sum"),
                 gross=("GROSS_SALE","sum"),
                 net=("NET_SALE","sum"))
            .reset_index())
print("\n📅 Weekly summary:")
display(weekly.head(10))

In [61]:
!pip freeze > requirements.txt